# Predicting Sale Price as the Response Variable

**Research Question:** Can a NYC property's sale value be predcited based of its square footage, location, and other characteristics?

**Response Variable:** Sale Price of NYC properties between 2000 and 2019

**Potential Predcitor Variables:** Square Footage, Number of Residential Units within a property, Number of Commenrcial units within a property, Borough property is located in, Year of Sale of the Property, Year the Property was built

## Data Load In

In [67]:
import os
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml import Pipeline
from pyspark.sql import functions as F
from pyspark.sql.functions import mean
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.regression import RandomForestRegressor
import matplotlib.pyplot as plt
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

In [68]:
spark = (
    SparkSession.builder
    .master("local[8]")
    .appName("nyc_sales_regression")
    .config("spark.driver.memory", "10g")
    .config("spark.driver.maxResultSize", "2g")
    .config("spark.sql.shuffle.partitions", "16")
    .config("spark.default.parallelism", "16")
    .getOrCreate()
)

In [69]:
df = spark.read.parquet("data/nyc_property_sales_cleaned")

In [70]:
df.count()

1600162

In [71]:
df = df.drop_duplicates()

In [72]:
df.count()

1571610

In [73]:
df.show(3)

+-------+------------+-----------------------+--------------------+-----+----+-------------------------+--------------------+----------------+--------+-----------------+----------------+-----------+----------------+-----------------+----------+-------------------------+------------------------------+----------+----------+----+------------+
|Borough|Neighborhood|Building Class Category|Tax Class At Present|Block| Lot|Building Class At Present|             Address|Apartment Number|Zip Code|Residential Units|Commercial Units|Total Units|Land Square Feet|Gross Square Feet|Year Built|Tax Class At Time Of Sale|Building Class At Time Of Sale|Sale Price| Sale Date|Year|Borough_Name|
+-------+------------+-----------------------+--------------------+-----+----+-------------------------+--------------------+----------------+--------+-----------------+----------------+-----------+----------------+-----------------+----------+-------------------------+------------------------------+----------+----

In [74]:
df.dtypes

[('Borough', 'int'),
 ('Neighborhood', 'string'),
 ('Building Class Category', 'string'),
 ('Tax Class At Present', 'string'),
 ('Block', 'string'),
 ('Lot', 'string'),
 ('Building Class At Present', 'string'),
 ('Address', 'string'),
 ('Apartment Number', 'string'),
 ('Zip Code', 'int'),
 ('Residential Units', 'int'),
 ('Commercial Units', 'int'),
 ('Total Units', 'int'),
 ('Land Square Feet', 'double'),
 ('Gross Square Feet', 'double'),
 ('Year Built', 'int'),
 ('Tax Class At Time Of Sale', 'string'),
 ('Building Class At Time Of Sale', 'string'),
 ('Sale Price', 'double'),
 ('Sale Date', 'date'),
 ('Year', 'int'),
 ('Borough_Name', 'string')]

In [75]:
df.select("Building Class Category").distinct().count()

99

In [76]:
df.select("Neighborhood").distinct().count()

267

In [77]:
df.select("Building Class At Time of Sale").distinct().count()

215

In [78]:
df.select("Tax Class At Time of Sale").distinct().count()

4

In [79]:
df.select("Zip Code").distinct().count()

242

## Data Preprocessing

In [80]:
df.select(F.max("Sale Price")).show()

+---------------+
|max(Sale Price)|
+---------------+
|  4.111111766E9|
+---------------+



In [81]:
df.select(F.min("Sale Price")).show()

+---------------+
|min(Sale Price)|
+---------------+
|            0.0|
+---------------+



In [82]:
top_5_df = df.select("Sale Price").orderBy(F.col("Sale Price").desc()).limit(50).show()

+-------------+
|   Sale Price|
+-------------+
|4.111111766E9|
|   4.040527E9|
|3.330132711E9|
|        2.8E9|
|2.397501899E9|
|       2.21E9|
|        2.2E9|
|      2.155E9|
|        2.1E9|
|     1.9329E9|
|        1.8E9|
|   1.791829E9|
|      1.775E9|
|       1.77E9|
|1.767913114E9|
|       1.76E9|
|1.739370834E9|
|       1.72E9|
|      1.575E9|
|       1.52E9|
+-------------+
only showing top 20 rows


In [83]:
df_high_value = df.filter(F.col("Sale Price") > 50000000)

In [84]:
df_high_value.count()

1968

There are 1968 properties in the data that have over $50 million as their sale price. When first doing the modeling for Sale Price, these large outliers led to predictions being far worse as they pulled the average predicted sale price up. The data will be filtered to get rid of properties of over $50 million dolalrs.

In [85]:
cat_cols = ["Borough_Name"]

num_cols = ["Residential Units","Commercial Units","Gross Square Feet"]

In [86]:
df = (
    df.filter(F.col("Sale Price") > 10000)
      .filter(F.col("Sale Price") < 50000000)
      .filter(F.col("Gross Square Feet") > 0)
      .filter(F.col("Land Square Feet") > 0)
      .filter(F.col("Year Built") > 1600)
      .dropna(subset=cat_cols + num_cols + ["Sale Price"])
)

In the above code, I am also removing all rows where the sale price is less than $10,000. This includes the roughly 30% of the sales that were only 0 dollars, as seen in the Exploratory Data Analysis. The properties with sale prices between $10,000 and $50,000,000 should for better model prediction, as the extreme price outliers are being left out on both sides. The filter also gets rid of any property that doesn't have square footage, or is built before 1600 to just eliminate any potentially weird or innacurate rows. Finally, the filter alos makes sure all rows left do not have NAs in the columns of borough name, residential units, commerical units, gross square feet, and sale price.

In [87]:
df.count()

538788

In [88]:
df.select(mean("Sale Price")).show()

+-----------------+
|  avg(Sale Price)|
+-----------------+
|950544.0619835631|
+-----------------+



### Reducing High Number of Category Predictors

The categorical variables "Building Class at Time of Sale" and "Building Class Category" have 194 and 82 categories respectviely after filtering the data down. 

"Building Class at Time of Sale" begins each category with a letter that correpsonds to a general definition for the property like condo or one-family home. So, we will be reducing that column to its first character to limit categories. 

"Building Class Category" begins each category with a 2 digit number like "01" or "32", where similarly to the previous column, the first 2 digits represent a category of properties like one-family homes, condos, and other specific real estate definitions like theatres. We will be reducing this column to its first 2 characters to limit categories.

In [89]:
df.select("Building Class at Time of Sale").distinct().count()

194

In [90]:
df.select("Building Class at Time of Sale").distinct().show(1000)

+------------------------------+
|Building Class at Time of Sale|
+------------------------------+
|                            A2|
|                            B3|
|                            S5|
|                            I5|
|                            E7|
|                            A6|
|                            K6|
|                            H6|
|                            K1|
|                            S4|
|                            C5|
|                            D5|
|                            M9|
|                            J8|
|                            Q9|
|                            U7|
|                            B9|
|                            A4|
|                            C9|
|                            J6|
|                            D2|
|                            L3|
|                            H4|
|                            C6|
|                            Z0|
|                            S3|
|                            K2|
|         

In [91]:
df = df.withColumn("Building Class at Time of Sale", F.col("Building Class at Time of Sale").substr(1, 1))

In [92]:
df.select("Building Class at Time of Sale").distinct().count()

25

In [93]:
df.select("Building Class Category").distinct().count()

82

In [94]:
df.select("Building Class Category").distinct().show(1000)

+-----------------------+
|Building Class Category|
+-----------------------+
|   01  ONE FAMILY HOMES|
|   02  TWO FAMILY HOMES|
|   32  HOSPITAL AND ...|
|         30  WAREHOUSES|
|           34  THEATRES|
|          27  FACTORIES|
|   09  COOPS - WALKU...|
|   13  CONDOS - ELEV...|
|      11A CONDO-RENTALS|
|   07  RENTALS - WAL...|
|   40  SELECTED GOVE...|
|      25  LUXURY HOTELS|
|   14  RENTALS - 4-1...|
|   36  OUTDOOR RECRE...|
|   08  RENTALS - ELE...|
|   31  COMMERCIAL VA...|
|   03  THREE FAMILY ...|
|   06  TAX CLASS 1 -...|
|   33  EDUCATIONAL F...|
|   35  INDOOR PUBLIC...|
|   37  RELIGIOUS FAC...|
|   41  TAX CLASS 4 -...|
|   18  TAX CLASS 3 -...|
|   21  OFFICE BUILDINGS|
|    22  STORE BUILDINGS|
|       26  OTHER HOTELS|
|            17  CONDOPS|
|   05  TAX CLASS 1 V...|
|   29  COMMERCIAL GA...|
|   38  ASYLUMS AND H...|
|     23  LOFT BUILDINGS|
|   10  COOPS - ELEVA...|
|   03  THREE FAMILY ...|
|   02  TWO FAMILY DW...|
|   11  SPECIAL CONDO...|
|   01  ONE 

In [95]:
df = df.withColumn("Building Class Category", F.col("Building Class Category").substr(1, 2))

In [96]:
df.select("Building Class Category").distinct().count()

42

# Data Modeling and Evaluation

## Baseline Linear Regression Model

In [97]:
cat_cols = ["Borough_Name"]

num_cols = ["Residential Units","Commercial Units","Gross Square Feet"]

In [98]:
indexers = [
    StringIndexer(
        inputCol=col,
        outputCol=col + "_idx"
    )
    for col in cat_cols
]


encoder = OneHotEncoder(
    inputCols=[col + "_idx" for col in cat_cols],
    outputCols=[col + "_ohe" for col in cat_cols],
    dropLast=True
)


assembler = VectorAssembler(
    inputCols=[f"{c}_ohe" for c in cat_cols] + num_cols,
    outputCol="features",
    handleInvalid="skip"
)

In [99]:
lr = LinearRegression(featuresCol="features", labelCol="Sale Price")

pipeline = Pipeline(stages=indexers + [encoder, assembler, lr])

train, test = df.randomSplit([0.8, 0.2], seed=101)
model = pipeline.fit(train)
preds = model.transform(test)

26/11/25 10:23:22 WARN Instrumentation: [4a4be94e] regParam is zero, which might cause numerical instability and overfitting.


In [100]:
evaluator = RegressionEvaluator(
    labelCol="Sale Price",
    predictionCol="prediction",
)

mse  = evaluator.setMetricName("mse").evaluate(preds)
mae  = evaluator.setMetricName("mae").evaluate(preds)
rmse = evaluator.setMetricName("rmse").evaluate(preds)
r2   = evaluator.setMetricName("r2").evaluate(preds)

print(f"MSE:  {mse:,.2f}")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R2:   {r2:.4f}")

MSE:  4,866,839,320,865.99
MAE:  686,963.62
RMSE: 2,206,091.41
R2:   0.1626


The results from the baseline linear regression model are poor, with only a 0.16 R-squared value. This indicates very little of the variability in Sale Price was able to be accounted for by our predictors. The MAE is $686,963.62, which is very high considering that the mean sale price after filtering is $950544 for the properties in the data.

## Random Forest Regression (Champion Model)

In [101]:
cat_cols = ["Borough_Name","Tax Class At Time of Sale","Neighborhood","Building Class at Time of Sale","Building Class Category","Zip Code"]

num_cols = ["Residential Units","Commercial Units","Gross Square Feet","Year Built","Year"]

In [102]:
train, test = df.randomSplit([0.8, 0.2], seed=101)


In [103]:

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

assembler = VectorAssembler(
    inputCols=[f"{c}_idx" for c in cat_cols] + num_cols,
    outputCol="features",
    handleInvalid="keep",   # avoids failures on nulls in numeric columns
)

rf = RandomForestRegressor(
    featuresCol="features", labelCol="Sale Price",
    numTrees=100, maxDepth=10, maxBins=300, seed=101,
)

pipeline = Pipeline(stages=indexers + [assembler, rf])


In [104]:
# Cross-validation setup
param_grid = (
    ParamGridBuilder()
    .addGrid(rf.maxDepth, [3, 5, 10])
    .addGrid(rf.numTrees, [100, 200])
    .addGrid(rf.minInstancesPerNode, [1, 3])
    .build()
)

cv_evaluator = RegressionEvaluator(
    labelCol="Sale Price", predictionCol="prediction", metricName="rmse"
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=cv_evaluator,
    numFolds=3,       
    parallelism=2,
    seed=101,
)

In [105]:
cv_model = cv.fit(train)

# CV results: average RMSE per parameter combo
for params, score in zip(param_grid, cv_model.avgMetrics):
    print({p.name: v for p, v in params.items()}, f"RMSE={score:,.0f}")

best_rf = cv_model.bestModel.stages[-1]
print("\nBest maxDepth:", best_rf.getMaxDepth())
print("Best numTrees:", best_rf.getNumTrees)
print("Best minInstancesPerNode:", best_rf.getMinInstancesPerNode())

26/11/25 10:23:45 WARN DAGScheduler: Broadcasting large task binary with size 1564.6 KiB
26/11/25 10:23:48 WARN DAGScheduler: Broadcasting large task binary with size 1566.8 KiB
26/11/25 10:23:55 WARN DAGScheduler: Broadcasting large task binary with size 1605.3 KiB
26/11/25 10:23:58 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
26/11/25 10:24:04 WARN DAGScheduler: Broadcasting large task binary with size 1605.3 KiB
26/11/25 10:24:06 WARN DAGScheduler: Broadcasting large task binary with size 2.9 MiB
26/11/25 10:24:11 WARN DAGScheduler: Broadcasting large task binary with size 1564.6 KiB
26/11/25 10:24:13 WARN DAGScheduler: Broadcasting large task binary with size 2.7 MiB
26/11/25 10:24:14 WARN DAGScheduler: Broadcasting large task binary with size 4.8 MiB
26/11/25 10:24:17 WARN DAGScheduler: Broadcasting large task binary with size 8.1 MiB
26/11/25 10:24:20 WARN DAGScheduler: Broadcasting large task binary with size 1122.1 KiB
26/11/25 10:24:21 WARN DAGScheduler:

{'maxDepth': 3, 'numTrees': 100, 'minInstancesPerNode': 1} RMSE=1,759,322
{'maxDepth': 3, 'numTrees': 100, 'minInstancesPerNode': 3} RMSE=1,759,322
{'maxDepth': 3, 'numTrees': 200, 'minInstancesPerNode': 1} RMSE=1,763,143
{'maxDepth': 3, 'numTrees': 200, 'minInstancesPerNode': 3} RMSE=1,763,143
{'maxDepth': 5, 'numTrees': 100, 'minInstancesPerNode': 1} RMSE=1,601,085
{'maxDepth': 5, 'numTrees': 100, 'minInstancesPerNode': 3} RMSE=1,600,006
{'maxDepth': 5, 'numTrees': 200, 'minInstancesPerNode': 1} RMSE=1,603,076
{'maxDepth': 5, 'numTrees': 200, 'minInstancesPerNode': 3} RMSE=1,602,155
{'maxDepth': 10, 'numTrees': 100, 'minInstancesPerNode': 1} RMSE=1,438,144
{'maxDepth': 10, 'numTrees': 100, 'minInstancesPerNode': 3} RMSE=1,429,121
{'maxDepth': 10, 'numTrees': 200, 'minInstancesPerNode': 1} RMSE=1,434,744
{'maxDepth': 10, 'numTrees': 200, 'minInstancesPerNode': 3} RMSE=1,425,902

Best maxDepth: 10
Best numTrees: 200
Best minInstancesPerNode: 3


In [106]:
# Final evaluation on the held-out test set
preds = cv_model.transform(test)

evaluator = RegressionEvaluator(labelCol="Sale Price", predictionCol="prediction")

mse  = evaluator.setMetricName("mse").evaluate(preds)
mae  = evaluator.setMetricName("mae").evaluate(preds)
rmse = evaluator.setMetricName("rmse").evaluate(preds)
r2   = evaluator.setMetricName("r2").evaluate(preds)

print(f"MSE:  {mse:,.2f}")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R2:   {r2:.4f}")

MSE:  2,005,799,605,920.87
MAE:  370,431.61
RMSE: 1,416,262.55
R2:   0.6549


The Random Forest Regression model did improve on performance comapred to the baseline linear regression. The R-squared score went up from 0.16 to 0.66, and the MAE dropped from $686,963.62 to $370,431.61. Still, the model struggles with large errors, as the RMSE is almost 4 times as large as the MAE.

## Gradient Boosted Trees

In [107]:
cat_cols = ["Borough_Name","Tax Class At Time of Sale","Neighborhood","Building Class at Time of Sale","Building Class Category","Zip Code"]

num_cols = ["Residential Units","Commercial Units","Gross Square Feet","Year Built","Year"]

In [108]:
train, test = df.randomSplit([0.8, 0.2], seed=101)


In [114]:
indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in cat_cols
]

assembler = VectorAssembler(
    inputCols=[f"{c}_idx" for c in cat_cols] + num_cols,
    outputCol="features",
    handleInvalid="keep",
)

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="Sale Price",
    maxIter=100,
    maxDepth=6,
    maxBins=300,
    stepSize=0.1,
    seed=101,
)

pipeline = Pipeline(stages=indexers + [assembler, gbt])

In [115]:
# Cross-validation setup
param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5])
    .addGrid(gbt.stepSize, [0.05, 0.1])
    .addGrid(gbt.maxIter, [100, 200])
    .build()
)

cv_evaluator = RegressionEvaluator(
    labelCol="Sale Price", predictionCol="prediction", metricName="rmse"
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=param_grid,
    evaluator=cv_evaluator,
    numFolds=3,
    parallelism=2,    
    seed=101,
)

In [116]:
cv_model = cv.fit(train)

# CV results: average RMSE per parameter combo
for params, score in zip(param_grid, cv_model.avgMetrics):
    print({p.name: v for p, v in params.items()}, f"RMSE={score:,.0f}")

best_gbt = cv_model.bestModel.stages[-1]
print("\nBest maxDepth:", best_gbt.getMaxDepth())
print("Best stepSize:", best_gbt.getStepSize())
print("Best maxIter:", best_gbt.getMaxIter())

26/11/25 10:44:04 WARN DAGScheduler: Broadcasting large task binary with size 1000.7 KiB
26/11/25 10:44:04 WARN DAGScheduler: Broadcasting large task binary with size 1003.8 KiB
26/11/25 10:44:05 WARN DAGScheduler: Broadcasting large task binary with size 1000.6 KiB
26/11/25 10:44:05 WARN DAGScheduler: Broadcasting large task binary with size 1004.8 KiB
26/11/25 10:44:05 WARN DAGScheduler: Broadcasting large task binary with size 1003.8 KiB
26/11/25 10:44:05 WARN DAGScheduler: Broadcasting large task binary with size 1008.3 KiB
26/11/25 10:44:05 WARN DAGScheduler: Broadcasting large task binary with size 1004.8 KiB
26/11/25 10:44:05 WARN DAGScheduler: Broadcasting large task binary with size 1013.7 KiB
26/11/25 10:44:05 WARN DAGScheduler: Broadcasting large task binary with size 1008.3 KiB
26/11/25 10:44:05 WARN DAGScheduler: Broadcasting large task binary with size 1014.1 KiB
26/11/25 10:44:05 WARN DAGScheduler: Broadcasting large task binary with size 1013.6 KiB
26/11/25 10:44:05 WAR

{'maxDepth': 3, 'stepSize': 0.05, 'maxIter': 100} RMSE=1,528,438
{'maxDepth': 3, 'stepSize': 0.05, 'maxIter': 200} RMSE=1,499,860
{'maxDepth': 3, 'stepSize': 0.1, 'maxIter': 100} RMSE=1,510,532
{'maxDepth': 3, 'stepSize': 0.1, 'maxIter': 200} RMSE=1,525,509
{'maxDepth': 5, 'stepSize': 0.05, 'maxIter': 100} RMSE=1,510,270
{'maxDepth': 5, 'stepSize': 0.05, 'maxIter': 200} RMSE=1,552,897
{'maxDepth': 5, 'stepSize': 0.1, 'maxIter': 100} RMSE=1,574,702
{'maxDepth': 5, 'stepSize': 0.1, 'maxIter': 200} RMSE=1,651,423

Best maxDepth: 3
Best stepSize: 0.05
Best maxIter: 200


In [117]:
# Final evaluation on the held-out test set
preds = cv_model.transform(test).cache()

evaluator = RegressionEvaluator(
    labelCol="Sale Price",
    predictionCol="prediction",
)

mse  = evaluator.setMetricName("mse").evaluate(preds)
mae  = evaluator.setMetricName("mae").evaluate(preds)
rmse = evaluator.setMetricName("rmse").evaluate(preds)
r2   = evaluator.setMetricName("r2").evaluate(preds)

print(f"MSE:  {mse:,.2f}")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R2:   {r2:.4f}")

MSE:  2,123,135,840,525.87
MAE:  386,337.55
RMSE: 1,457,098.43
R2:   0.6347


The Gradient Boosted Regresion Tree model performed worse compared to the Random Forest Regression Model. Compared to the previous model, the R-squared decreased from 0.66 to 0.63, and the MAE increased from $370,431.61 to $386,337.55.


# Final Conclusion for Predicting Sale Price

Even with information on a property that includes its square footage, building classification, location within New York, and other notable predictors; predciting Sale Price is still a challening task. The Random Forest Regression Model and Gradient Boosted Tree Regresion Model did improve upon a Linear Regression Model with only a few predictors, but these models still are on average over $370,000 from the true Sale Price of properites in the test set. Clearly, more is at play when deciding the price, as well as larger prices even after the filtering could be bring the average predcited prices up.